# EvaNet (elevation-guided U-Net)

A separate model from the ResNet50 U-Nets. It uses **EvaNet** ([paper code](https://github.com/MTSami/EvaNet), MIT license), copied into `scripts/evanet/`. See the README there for the source commit and the one bug fix.

How EvaNet differs from `unet_dem.ipynb`:
- **Elevation is its own stream through the whole network.** Every conv layer computes `conv(rgb) * sigmoid(conv(elevation))`, so the DEM decides per pixel how much image information gets through. It isn't just a 4th input channel.
- **Gravity loss.** Pixels are compared with their 3x3 neighbours. If a neighbour is flooded and this pixel is lower, it should be flooded. If a neighbour is dry and this pixel is higher, it should be dry. Unknown pixels are ignored. The paper found this loss alone beat cross-entropy and cross-entropy + gravity.

It uses the same splits and IoU as `unet_dem.ipynb`. With `USE_EVANET_REGIONS = True`, training also gets EvaNet's 3 North Carolina regions (`scripts/evanet_data.py`). Val and test are unchanged, so the results can still be compared directly.

**Kernel:** `EvaNet (PyTorch)`. Create the environment with `environment-evanet.yml`.

In [ ]:
import sys
import numpy as np
import torch

sys.path.append("../scripts")
from dataset import get_tiles, SPLITS
from evanet import EvaNet, ElevationLoss

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

# EvaNet creates its conv layers on the GPU when one is available, so the whole model has to live there
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")

In [ ]:
MAX_UNKNOWN = 1.0  # same as unet_dem.ipynb: 1.0 = keep every training tile

# "train" = scenes 1, 2, 3, 8, 9 (same as unet_dem.ipynb)
# "train_plus_new" = also the later-labeled scenes 13-17 and 19 (Florence NC, Harvey west Houston)
TRAIN_SPLIT = "train"

# Add EvaNet's 3 North Carolina regions (Hurricane Matthew 2016) to the TRAINING set only.
# Val and test stay the project's Texas scenes, so results remain comparable with unet_dem.ipynb.
# Set to False to train on exactly the same tiles as the other models.
USE_EVANET_REGIONS = True

In [ ]:
# Same tiles as unet_dem.ipynb: train overlaps by half, val and test keep every tile.
# Training data is a list of (X, y) arrays that are never concatenated, which would briefly need twice the RAM.
train_parts = [get_tiles(SPLITS[TRAIN_SPLIT], stride=128, max_unknown=MAX_UNKNOWN)]
if USE_EVANET_REGIONS:
    from evanet_data import get_region_tiles
    train_parts.append(get_region_tiles(stride=128, max_unknown=MAX_UNKNOWN))
X_val, y_val = get_tiles(SPLITS["val"])
X_test, y_test = get_tiles(SPLITS["test"])

for name, X, y in [*[(f"Train{k}", *p) for k, p in enumerate(train_parts)], ("Val", X_val, y_val), ("Test", X_test, y_test)]:
    known = y[y != -1]
    print(f"{name:6}  X {X.shape}  unknown {(y == -1).mean():.1%}  flood (of known) {known.mean():.1%}")
print(f"training split: {TRAIN_SPLIT} {SPLITS[TRAIN_SPLIT]}  regions: {USE_EVANET_REGIONS}  "
      f"tiles: {sum(len(X) for X, _ in train_parts)}")

## Convert tiles to EvaNet's inputs

`dataset.py` gives `(256, 256, 4)` tiles: RGB from 0 to 1 plus the DEM minus the image's mean elevation. EvaNet wants channels first and three separate tensors:

| EvaNet input | Made from | Used by |
|---|---|---|
| `rgb` (3, H, W) | RGB scaled to -1 to 1: `(x - 0.5) / 0.5`, same as upstream | model |
| `norm_elev` (1, H, W) | DEM scaled to 0-1 with the **training set's** min/max (val/test clipped) | model |
| `elev` (1, H, W) | DEM unscaled, in metres | loss (it only uses height differences, so mean-centred is fine) |
| `labels` (1, H, W) | 1 = flood becomes **+1**, 0 = dry becomes **-1**, -1 = unknown becomes **0** | loss |

Conversion happens one tile at a time in `__getitem__`, so the full arrays aren't copied. Training tiles get the same random rot90 and flip as `unet_dem.ipynb`.

In [ ]:
from torch.utils.data import ConcatDataset, Dataset, DataLoader

ELEV_MIN = float(min(X[..., 3].min() for X, _ in train_parts))
ELEV_MAX = float(max(X[..., 3].max() for X, _ in train_parts))
print(f"Training DEM range (mean-centred): {ELEV_MIN:.2f} to {ELEV_MAX:.2f} m")


class EvaNetTiles(Dataset):
    """Wraps the (N, H, W, 4) tiles and (N, H, W) labels from dataset.py for EvaNet."""

    def __init__(self, X, y, augment=False):
        self.X, self.y, self.augment = X, y, augment

    def __len__(self):
        return len(self.X)

    def __getitem__(self, i):
        x = torch.from_numpy(self.X[i]).permute(2, 0, 1)        # (4, H, W)
        y = torch.from_numpy(self.y[i].astype(np.float32))[None]  # (1, H, W): 1 / 0 / -1

        if self.augment:  # same augmentation as unet_dem.ipynb, applied to every channel and the label
            k = int(torch.randint(0, 4, ()))
            x, y = torch.rot90(x, k, (1, 2)), torch.rot90(y, k, (1, 2))
            if torch.rand(()) < 0.5:
                x, y = torch.flip(x, (2,)), torch.flip(y, (2,))

        rgb = (x[:3] - 0.5) / 0.5
        elev = x[3:4]
        norm_elev = ((elev - ELEV_MIN) / (ELEV_MAX - ELEV_MIN)).clamp(0, 1)
        labels = torch.where(y == 1, 1.0, torch.where(y == 0, -1.0, 0.0))  # flood +1, dry -1, unknown 0

        return {"rgb": rgb.contiguous(), "norm_elev": norm_elev.contiguous(),
                "elev": elev.contiguous(), "labels": labels.contiguous(), "y": y.contiguous()}


BATCH_SIZE = 4  # same as unet_dem.ipynb and upstream EvaNet

train_ds = ConcatDataset([EvaNetTiles(X, y, augment=True) for X, y in train_parts])
val_ds = EvaNetTiles(X_val, y_val)
test_ds = EvaNetTiles(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

print(f"batches: train {len(train_loader)}  val {len(val_loader)}  test {len(test_loader)}")

In [ ]:
# Check one batch: shapes and value ranges
batch = next(iter(train_loader))
for k, v in batch.items():
    print(f"{k:9} {tuple(v.shape)}  {v.dtype}  min {v.min():.3f}  max {v.max():.3f}")

# Labels must line up: flood (+1) where y == 1, dry (-1) where y == 0, unknown (0) where y == -1
assert torch.equal(batch["labels"] == 1, batch["y"] == 1)
assert torch.equal(batch["labels"] == -1, batch["y"] == 0)
assert torch.equal(batch["labels"] == 0, batch["y"] == -1)
print("label conversion OK")

## Build the model

`ultrasmall=True` is the configuration upstream actually trains: 3 down and 3 up levels, 8 to 64 channels. The larger option has an argument bug upstream and crashes.

**Output channel 0 is flood** and channel 1 is dry, so the flood mask is `pred.argmax(1) == 0`.

In [ ]:
model = EvaNet(BATCH_SIZE, n_channels=3, n_classes=2, ultrasmall=True, device=DEVICE).to(DEVICE)
loss_fn = ElevationLoss()

n_params = sum(p.numel() for p in model.parameters())
print(f"EvaNet parameters: {n_params:,}")


def predict_flood(pred):
    """(B, 2, H, W) EvaNet output -> (B, H, W) bool flood mask. Channel 0 is flood."""
    return pred.argmax(1) == 0

## Sanity check: one forward and backward pass

Run a single batch through the model and the gravity loss, then backpropagate. This checks shapes, that the loss is finite, and that every layer gets a gradient. Nothing is trained here.

In [ ]:
model.train()
rgb = batch["rgb"].to(DEVICE)
norm_elev = batch["norm_elev"].to(DEVICE)
elev = batch["elev"].to(DEVICE)
labels = batch["labels"].to(DEVICE)

pred = model(rgb, norm_elev)
loss = loss_fn(pred, elev, labels)
model.zero_grad()
loss.backward()

no_grad = [n for n, p in model.named_parameters() if p.requires_grad and p.grad is None]
bad_grad = [n for n, p in model.named_parameters() if p.grad is not None and not torch.isfinite(p.grad).all()]

print("input:", tuple(rgb.shape), tuple(norm_elev.shape), "-> output:", tuple(pred.shape))
print(f"output range: {pred.min():.3f} to {pred.max():.3f} (sigmoid)")
print(f"loss: {loss.item():.1f}  (a sum over pixels, so it is large)")
print(f"predicted flood (untrained): {predict_flood(pred).float().mean():.1%}")
print("params without a gradient:", no_grad or "none")
print("params with NaN/inf gradient:", bad_grad or "none")

assert pred.shape == (BATCH_SIZE, 2, 256, 256)
assert torch.isfinite(loss) and not bad_grad
print("ready to train")

## Training

Two configurations:
- **`adam`** (default, practical): Adam, lr 1e-4. It converges in tens of epochs.
- **`sgd`** (faithful to upstream): SGD, lr 1e-6, about 350 epochs. Very slow.

Each epoch reports training loss, validation loss and validation flood/dry IoU (image 12), with unknown pixels ignored as in `unet_dem.ipynb`. The best epoch **by validation loss** (the same rule as upstream and `unet_dem`) is saved to `../outputs/<RUN_NAME>.pt`, and the history to `../outputs/<RUN_NAME>_history.json`. Training stops early after `PATIENCE` epochs without improvement.

Loss values are **per tile**. The gravity loss is a sum over pixels, so they're large; only compare them within the same setup.

In [ ]:
from pathlib import Path

OPTIMIZER = "adam"   # "adam" (practical) or "sgd" (upstream: lr 1e-6, ~350 epochs)
LR = 1e-4 if OPTIMIZER == "adam" else 1e-6
EPOCHS = 60
PATIENCE = 10        # stop after this many epochs without a better val loss
MAX_TRAIN_BATCHES = None  # set to a small number for a quick test run; None = full epochs

RUN_NAME = (f"evanet_{OPTIMIZER}_lr{LR:g}" + ("_regions" if USE_EVANET_REGIONS else "")
            + ("_plusnew" if TRAIN_SPLIT == "train_plus_new" else ""))
ckpt_path = Path(f"../outputs/{RUN_NAME}.pt")
history_path = Path(f"../outputs/{RUN_NAME}_history.json")
print("run:", RUN_NAME)

In [ ]:
def iou_counts(pred_flood, y):
    """Pixel counts over known pixels: flood inter/union, dry inter/union, predicted flood, known.
    y is (B, 1, H, W) with 1 / 0 / -1."""
    y = y[:, 0]
    known, true_flood, true_dry = y != -1, y == 1, y == 0
    pred_dry = ~pred_flood
    return np.array([
        (pred_flood & true_flood & known).sum().item(), ((pred_flood | true_flood) & known).sum().item(),
        (pred_dry & true_dry & known).sum().item(), ((pred_dry | true_dry) & known).sum().item(),
        (pred_flood & known).sum().item(), known.sum().item(),
    ], dtype=np.int64)


@torch.no_grad()
def evaluate(loader):
    """Returns mean loss per tile, flood IoU, dry IoU and the share of known pixels predicted flood."""
    model.eval()
    total_loss, n_tiles, c = 0.0, 0, np.zeros(6, np.int64)
    for b in loader:
        pred = model(b["rgb"].to(DEVICE), b["norm_elev"].to(DEVICE))
        total_loss += loss_fn(pred, b["elev"].to(DEVICE), b["labels"].to(DEVICE)).item()
        n_tiles += len(pred)
        c += iou_counts(predict_flood(pred).cpu(), b["y"])
    return total_loss / n_tiles, c[0] / c[1], c[2] / c[3], c[4] / c[5]

In [ ]:
import json
import time

# Fresh model, so this cell can be re-run on its own
torch.manual_seed(SEED)
model = EvaNet(BATCH_SIZE, n_channels=3, n_classes=2, ultrasmall=True, device=DEVICE).to(DEVICE)
if OPTIMIZER == "adam":
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
else:
    optimizer = torch.optim.SGD(model.parameters(), lr=LR)

ckpt_path.parent.mkdir(exist_ok=True)
history = {"run": RUN_NAME, "lr": LR, "optimizer": OPTIMIZER, "use_evanet_regions": USE_EVANET_REGIONS,
           "elev_min": ELEV_MIN, "elev_max": ELEV_MAX, "epochs": []}
best_val, best_epoch = float("inf"), 0

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    model.train()
    train_loss, n_tiles = 0.0, 0
    for step, b in enumerate(train_loader):
        if MAX_TRAIN_BATCHES is not None and step >= MAX_TRAIN_BATCHES:
            break
        pred = model(b["rgb"].to(DEVICE), b["norm_elev"].to(DEVICE))
        loss = loss_fn(pred, b["elev"].to(DEVICE), b["labels"].to(DEVICE))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        n_tiles += len(pred)
    train_loss /= n_tiles

    val_loss, val_iou_flood, val_iou_dry, val_says_flood = evaluate(val_loader)
    improved = val_loss < best_val
    if improved:
        best_val, best_epoch = val_loss, epoch
        torch.save({"model": model.state_dict(), "epoch": epoch, "val_loss": val_loss,
                    "elev_min": ELEV_MIN, "elev_max": ELEV_MAX, "run": RUN_NAME}, ckpt_path)

    history["epochs"].append({"epoch": epoch, "train_loss": train_loss, "val_loss": val_loss,
                              "val_iou_flood": val_iou_flood, "val_iou_dry": val_iou_dry,
                              "val_says_flood": val_says_flood, "seconds": time.time() - t0})
    history_path.write_text(json.dumps(history, indent=1))
    print(f"epoch {epoch:3}/{EPOCHS}  train {train_loss:9.0f}  val {val_loss:9.0f}  "
          f"val IoU flood {val_iou_flood:.3f} dry {val_iou_dry:.3f}  says flood {val_says_flood:.1%}  "
          f"{time.time() - t0:5.0f}s{'  *saved' if improved else ''}", flush=True)

    if epoch - best_epoch >= PATIENCE:
        print(f"no improvement for {PATIENCE} epochs, stopping")
        break

print(f"best epoch {best_epoch} (val loss {best_val:.0f}) saved to {ckpt_path}")

In [ ]:
import json
import matplotlib.pyplot as plt

saved = json.loads(history_path.read_text())["epochs"]
best_epoch = min(saved, key=lambda e: e["val_loss"])["epoch"]
ep = [e["epoch"] for e in saved]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(ep, [e["train_loss"] for e in saved], label="train")
ax1.plot(ep, [e["val_loss"] for e in saved], label="val")
ax1.axvline(best_epoch, color="gray", ls="--", lw=1, label=f"best (epoch {best_epoch})")
ax1.set(title="Gravity loss per tile", xlabel="epoch")
ax1.legend()
ax2.plot(ep, [e["val_iou_flood"] for e in saved], label="flood IoU")
ax2.plot(ep, [e["val_iou_dry"] for e in saved], label="dry IoU")
ax2.axvline(best_epoch, color="gray", ls="--", lw=1)
ax2.set(title="Validation IoU (image 12)", xlabel="epoch", ylim=(0, 1))
ax2.legend()
plt.tight_layout()
plt.show()

## Test results

Load the best checkpoint and score the test images (4, 5, 6, 7, 10, 11) with the same flood/dry IoU as `unet_dem.ipynb`: first over all test tiles together, then per image. EvaNet picks a class with argmax, so there's no cutoff to tune.

In [ ]:
ckpt = torch.load(ckpt_path, map_location=DEVICE)
model.load_state_dict(ckpt["model"])
ELEV_MIN, ELEV_MAX = ckpt["elev_min"], ckpt["elev_max"]  # scale the DEM the same way as during training
print(f"loaded {ckpt_path.name}: epoch {ckpt['epoch']}, val loss {ckpt['val_loss']:.0f}")

for name, loader, y in [("VAL", val_loader, y_val), ("TEST", test_loader, y_test)]:
    _, f, d, says = evaluate(loader)
    print(f"{name:4}  flood IoU {f:.3f}  dry IoU {d:.3f}  says flood {says:.1%}  label {(y[y != -1] == 1).mean():.1%}")

print()
results = {}
# Test images, then the held-out Ida scene (never trained, validated or tuned on)
for i in SPLITS["test"] + SPLITS["ida_hard_case"]:
    Xi, yi = get_tiles([i])
    _, f, d, says = evaluate(DataLoader(EvaNetTiles(Xi, yi), batch_size=BATCH_SIZE))
    results[i] = {"flood_iou": f, "dry_iou": d, "says_flood": says}
    known = yi != -1
    tag = "  (Ida hard case)" if i in SPLITS["ida_hard_case"] else ""
    print(f"image {i:2}: flood IoU {f:.3f}  dry IoU {d:.3f}  says flood {says:.1%}  label {(yi[known] == 1).mean():.1%}{tag}")